In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'data'

files = sorted(RAW_DATA_PATH.glob('*.h5'))
print(len(files), 'files')
print(files[0].name)

In [ ]:
f = h5py.File(files[0], 'r')

for name, obj in f.items():
    if isinstance(obj, h5py.Dataset):
        print(f'{name:24} {str(obj.shape):22} {obj.dtype}')

In [ ]:
for k, v in f.attrs.items():
    print(f'{k:52} {v}')

In [ ]:
lut_names = [n for n in f if n.endswith(('_TEMP', '_RADIANCE', '_ALBEDO'))]

for n in lut_names:
    lut = f[n][:]
    units = f[n].attrs.get('units', b'').decode() if isinstance(f[n].attrs.get('units'), bytes) else f[n].attrs.get('units', '')
    print(f'{n:22} n={lut.shape[0]:<6} {str(units):24} {lut.min():.3f} .. {lut.max():.3f}')

In [ ]:
CHANNEL = 'TIR1'

lut = f[f'IMG_{CHANNEL}_TEMP'][:]
step = np.abs(np.diff(lut))

print('dtype       ', lut.dtype)
print('entries     ', lut.shape[0])
print('lut[0]      ', lut[0])
print('lut[-1]     ', lut[-1])
print('flat ends   ', int(np.sum(step == 0)))
print('monotonic   ', np.all(np.diff(lut)[np.diff(lut) != 0] < 0) or np.all(np.diff(lut)[np.diff(lut) != 0] > 0))
print('step min    ', step[step > 0].min())
print('step median ', np.median(step[step > 0]))
print('step max    ', step.max())

In [ ]:
print(lut[:10])
print(lut[-10:])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(lut)
ax[0].set_xlabel('count')
ax[0].set_ylabel('brightness temperature (K)')
ax[0].set_title(f'IMG_{CHANNEL}_TEMP')

ax[1].plot(step)
ax[1].set_xlabel('count')
ax[1].set_ylabel('step (K)')
ax[1].set_title('resolution per count')

plt.tight_layout()
plt.show()

In [ ]:
raw = f[f'IMG_{CHANNEL}'][0]
bt = lut[raw]

print('counts  ', raw.dtype, raw.shape, raw.min(), '..', raw.max())
print('bt      ', bt.dtype, bt.shape, f'{bt.min():.2f} .. {bt.max():.2f}')
print('distinct', np.unique(raw).size)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

im = ax[0].imshow(bt, cmap='gray_r')
ax[0].set_title(f'{CHANNEL} brightness temperature (K)')
plt.colorbar(im, ax=ax[0], fraction=0.046)

ax[1].hist(bt.ravel(), bins=256)
ax[1].set_xlabel('K')
ax[1].set_title('distribution')

plt.tight_layout()
plt.show()

In [ ]:
f.close()

In [ ]:
N = 20
CHANNELS = ['TIR1', 'TIR2', 'WV', 'MIR']

luts = {c: [] for c in CHANNELS}
for p in files[:N]:
    with h5py.File(p, 'r') as g:
        for c in CHANNELS:
            luts[c].append(g[f'IMG_{c}_TEMP'][:])

for c in CHANNELS:
    stack = np.stack(luts[c])
    spread = stack.max(axis=0) - stack.min(axis=0)
    print(f'{c:6} max spread across {N} files: {spread.max():.3f} K at count {spread.argmax()}')

In [ ]:
fig, ax = plt.subplots(1, len(CHANNELS), figsize=(16, 3.5), sharex=True)

for a, c in zip(ax, CHANNELS):
    stack = np.stack(luts[c])
    a.plot((stack - stack.mean(axis=0)).T, lw=0.6)
    a.set_title(c)
    a.set_xlabel('count')

ax[0].set_ylabel('deviation from mean LUT (K)')
plt.tight_layout()
plt.show()